# Adaptive Semantic JSCC — Stage 1 Colab Notebook

**Run on Google Colab with T4 GPU** (Runtime → Change runtime type → T4 GPU)

This notebook:
1. Clones your GitHub repo
2. Installs requirements
3. Runs channel unit tests
4. Trains Model 1 (Plain DeepJSCC)
5. Prints PSNR at SNR = 0, 5, 10, 15, 20 dB

**Stage gate:** PSNR must rise as SNR rises. If not, debug before Stage 2.

In [ ]:
# ── CELL 1: Verify GPU ──────────────────────────────────────────────────────
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Training will be very slow.')
    print('Go to Runtime → Change runtime type → Hardware accelerator: T4 GPU')

In [ ]:
# ── CELL 2: Clone repo ──────────────────────────────────────────────────────
# UPDATE THIS with your GitHub repo URL before running
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'
WORK_DIR = '/content/data_transmission'

import os
if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull

%cd {WORK_DIR}
!ls -la

In [ ]:
# ── CELL 3: Install requirements ────────────────────────────────────────────
!pip install -q -r requirements.txt
print('Requirements installed ✓')

In [ ]:
# ── CELL 4: Run channel unit tests ──────────────────────────────────────────
# All 5 tests must pass before any training.
!python channels.py

In [ ]:
# ── CELL 5: Sanity-check data loading ───────────────────────────────────────
# Downloads EuroSAT (~90 MB) on first run.
!python data.py

In [ ]:
# ── CELL 6: Smoke-test Model 1 architecture ─────────────────────────────────
!python models/deepjscc.py

In [ ]:
# ── CELL 7: Train Model 1 ───────────────────────────────────────────────────
# Expected time on T4 GPU: ~30-60 minutes for 80 epochs
# You will see per-epoch PSNR printed to confirm training is progressing.
!python train.py --config configs/model1.yaml

In [ ]:
# ── CELL 8: Evaluate Model 1 on test set ────────────────────────────────────
import sys, os
sys.path.insert(0, '/content/data_transmission')
os.chdir('/content/data_transmission')

import torch
import torch.nn as nn
from data  import get_loaders
from train import load_config, mse_to_psnr, _needs_imgnet, build_model

cfg    = load_config('configs/model1.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

_, _, test_loader = get_loaders(
    root='./data', batch_size=64, num_workers=2, download=True
)

model = build_model(cfg, device)
ckpt  = torch.load('checkpoints/model1/best.pt', map_location=device)
model.load_state_dict(ckpt['model'])
model.eval()
print(f'Loaded best checkpoint (epoch {ckpt["epoch"]+1})')

snrs     = [0, 5, 10, 15, 20]
channels = ['awgn', 'rayleigh']
criterion = nn.MSELoss()
results_table = {}

print()
print('Model 1 — Plain DeepJSCC  [Test Set]')
print(f'{"Channel":<12} {"SNR (dB)":>8} {"PSNR (dB)":>12}')
print('-' * 35)

with torch.no_grad():
    for ch in channels:
        psnrs = []
        for snr_db in snrs:
            total_mse, n = 0.0, 0
            for img_imgnet, img_unit, _ in test_loader:
                img_unit = img_unit.to(device)
                recon, _ = model(img_unit, channel=ch, snr_db=float(snr_db))
                total_mse += criterion(recon, img_unit).item()
                n += 1
            psnr = mse_to_psnr(total_mse / n)
            psnrs.append(psnr)
            print(f'{ch.upper():<12} {snr_db:>8} {psnr:>12.2f}')
        results_table[ch] = psnrs
        print()

In [ ]:
# ── CELL 9: Stage gate check ─────────────────────────────────────────────────
# PSNR MUST increase as SNR increases (higher SNR = less noise = better image)
# If this check fails, do NOT proceed to Stage 2.

print('=' * 55)
print('  STAGE GATE: PSNR must be monotonically increasing with SNR')
print('=' * 55)

gate_passed = True
for ch in channels:
    psnrs = results_table[ch]
    snr_pairs = list(zip(snrs, psnrs))
    monotone  = all(psnrs[i] <= psnrs[i+1] for i in range(len(psnrs)-1))
    status    = '✓ PASS' if monotone else '✗ FAIL'
    print(f'  {ch.upper():8s}: {status}')
    print(f'    SNR  : {snrs}')
    print(f'    PSNR : {[f"{p:.2f}" for p in psnrs]}')
    if not monotone:
        gate_passed = False
    print()

if gate_passed:
    print('🎉 Stage 1 COMPLETE! Proceed to Stage 2 (Model 2: ResNet-18 + DeepJSCC).')
else:
    print('⚠️  Stage gate FAILED. Do NOT proceed to Stage 2.')
    print('Debugging checklist:')
    print('  1. Run: python channels.py  — all 5 tests must pass')
    print('  2. Check train loss is decreasing (see results/model1/train_log.csv)')
    print('  3. Try increasing epochs to 100 or reducing lr to 5e-4')
    print('  4. Confirm img_unit is [0,1] and recon is [0,1] (sigmoid output)')

In [ ]:
# ── CELL 10 (optional): Quick visual sanity check ────────────────────────────
# Show original vs reconstructed images at SNR = 10 dB (AWGN)

import matplotlib.pyplot as plt
import numpy as np

model.eval()
img_imgnet, img_unit, labels = next(iter(test_loader))
img_unit_dev = img_unit[:5].to(device)

with torch.no_grad():
    recon_10dB, _ = model(img_unit_dev, channel='awgn', snr_db=10.0)
    recon_0dB,  _ = model(img_unit_dev, channel='awgn', snr_db=0.0)

def to_np(t):
    return t.cpu().permute(0, 2, 3, 1).numpy().clip(0, 1)

orig  = to_np(img_unit[:5])
r10   = to_np(recon_10dB)
r0    = to_np(recon_0dB)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
fig.suptitle('Model 1 (Plain DeepJSCC) — AWGN Channel', fontsize=14)

row_labels = ['Original', 'Recon @ 10 dB', 'Recon @  0 dB']
for row, (imgs, label) in enumerate(zip([orig, r10, r0], row_labels)):
    axes[row, 0].set_ylabel(label, fontsize=11, rotation=90, labelpad=10)
    for col in range(5):
        axes[row, col].imshow(imgs[col])
        axes[row, col].axis('off')

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/model1_visual_check.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: results/model1_visual_check.png')